In [1]:
pip install pandas nltk transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 33.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import wordnet
from transformers import pipeline
from tqdm import tqdm

# Download resource cần thiết
nltk.download('punkt')
nltk.download('wordnet')

# Load model BERT
unmasker = pipeline('fill-mask', model='bert-base-uncased')

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().replace('_', ' '))
    return list(synonyms)

def word_substitution(text, top_k=10):
    words = nltk.word_tokenize(text)
    freq_dist = nltk.FreqDist(words)
    top_words = [word.lower() for word, _ in freq_dist.most_common(top_k)]

    perturbed_words = words.copy()

    for i, word in enumerate(words):
        if word.lower() in top_words:
            synonyms = get_synonyms(word)
            if synonyms:
                perturbed_words[i] = synonyms[0]
            else:
                masked_words = words.copy()
                masked_words[i] = '[MASK]'
                masked_text = ' '.join(masked_words)
                result = unmasker(masked_text)
                if result:
                    perturbed_words[i] = result[0]['token_str']

    return ' '.join(perturbed_words)

# Đọc file dữ liệu
df = pd.read_csv('ielts_dataset_v1.4_rawxparaphrased.csv')

# Xử lý progress bar
tqdm.pandas()

# Áp dụng word_substitution có điều kiện
def conditional_perturb(row):
    if row['is_ai'] == 1 and row['variant'] == 'raw':
        return word_substitution(row['text'])
    else:
        return row['text']

# Apply với progress bar
df['perturbed_text'] = df.progress_apply(conditional_perturb, axis=1)

# Lưu file kết quả
df.to_csv('perturbed_essays.csv', index=False)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
  0%|          | 0/2729 [00:00<?, ?it/s]